# Multiclass IMV — Nursery

One-vs-rest IMV and the pairwise IMV matrix on the UCI Nursery dataset.

Following the original analysis we drop the two negligible classes
(`recommend`, `very_recom`, together under 2% of rows) and keep a three-class
problem. That is a modelling choice and is stated rather than implied.

**Pipeline**: download from UCI → ordinal-encode → 10 seeds × 3 estimators →
aggregate.

Full documentation: `documentation/examples/multi_imv/nursery.md`.

In [ ]:
import os, sys, tempfile, warnings, itertools
from pathlib import Path

_PATH_DISPLAY_ROOT = Path()

def relative_path(path, *, start=None):
    """Return a display-only path relative to an explicit local anchor."""
    anchor = Path(start) if start is not None else _PATH_DISPLAY_ROOT
    try:
        relative = os.path.relpath(Path(path).expanduser().resolve(),
                                   start=anchor.expanduser().resolve())
    except ValueError:  # Paths on different Windows drives cannot be relativized.
        relative = Path(path).name
    return Path(relative).as_posix()

def _relative_warning_text(message):
    text = str(message)
    roots = {Path.home(), Path(sys.prefix), Path(sys.base_prefix),
             _PATH_DISPLAY_ROOT, Path(tempfile.gettempdir())}
    for root in sorted(roots, key=lambda value: len(str(value)), reverse=True):
        absolute = str(root.expanduser().resolve())
        text = text.replace(absolute, relative_path(absolute))
    return text

def _show_relative_warning(message, category, filename, lineno, file=None, line=None):
    stream = file if file is not None else sys.stderr
    warning_text = _relative_warning_text(message)
    print(f"{relative_path(filename)}:{lineno}: {category.__name__}: {warning_text}", file=stream)

warnings.showwarning = _show_relative_warning

%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# No dataset is stored in this repository. Everything downloads on demand into a
# user-level cache outside the working tree.
CACHE = Path(os.environ.get("IMV_CACHE_HOME", Path.home() / ".cache" / "imv"))
CACHE.mkdir(parents=True, exist_ok=True)

ARTIFACTS = Path(os.environ.get("IMV_ARTIFACT_CACHE", CACHE / "notebook_artifacts")) / "multi_imv_nursery"
RESULTS = ARTIFACTS / "results"; RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES = ARTIFACTS / "figures"; FIGURES.mkdir(parents=True, exist_ok=True)

# Ten seeds, as required for any reported IMV result. The seed drives the fold
# split and the estimator, so every number below is a mean over ten independent
# fold partitions.
SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
N_SPLITS = 5

import imvpy
from imvpy import MulticlassIMV
from imvpy.utils import save_figure

IMVPY_SOURCE = Path(imvpy.__file__).resolve().parent
print(f"imvpy {imvpy.__version__} from {relative_path(IMVPY_SOURCE)}")


## 1. Download

Nothing is read from the repository.

In [ ]:
from ucimlrepo import fetch_ucirepo
DATASET, TITLE = "nursery", "Nursery"
raw = fetch_ucirepo(id=76)
frame = raw.data.features.copy()
frame["target_raw"] = raw.data.targets.iloc[:, 0].astype(str).str.strip()
print(frame.shape); print(frame["target_raw"].value_counts().to_dict())
frame.head()

## 2. Preprocess

In [ ]:
frame = frame[~frame["target_raw"].isin(["recommend", "very_recom"])].copy()
CLASS_NAMES = ["not_recom", "priority", "spec_prior"]
frame["target"] = frame["target_raw"].map({n: i for i, n in enumerate(CLASS_NAMES)})
FEATURES = [c for c in frame.columns if c not in ("target", "target_raw")]
for column in FEATURES:            # ordinal codes; all columns are categorical
    frame[column] = pd.Categorical(frame[column].astype(str)).codes
data = frame[FEATURES + ["target"]].dropna()
data["target"] = data["target"].astype(int)
print(data.shape, FEATURES); print(data["target"].value_counts().sort_index().to_dict())
data.head()

## 3. Estimators

In [ ]:
def model_factories(seed):
    """Three estimator families, deliberately small so the exact power set is affordable."""
    return {
        "logistic_regression": lambda: LogisticRegression(max_iter=5000, random_state=seed),
        "xgboost": lambda: XGBClassifier(
            n_estimators=60, max_depth=3, learning_rate=0.2, random_state=seed,
            verbosity=0, tree_method="hist", n_jobs=1),
        "lightgbm": lambda: LGBMClassifier(
            n_estimators=60, max_depth=3, learning_rate=0.2, random_state=seed,
            verbose=-1, n_jobs=1),
    }


## 4. One-vs-rest and pairwise IMV over ten seeds

Stratified folds are used so no fold can omit a class; the evaluator passes `model.classes_` through, so probability columns are matched to labels rather than to fold position.

In [ ]:
ova_records, matrices = [], {}
for seed in SEEDS:
    frame = data.reset_index(drop=True)
    for name, factory in model_factories(seed).items():
        evaluator = MulticlassIMV(
            frame, "target", factory, n_splits=N_SPLITS,
            optional_explanatory_variables=FEATURES,
            random_state=seed, stratified=True,
        )
        _, ova_mean = evaluator.k_fold_one_vs_all()
        _, matrix_mean = evaluator.k_fold_imv_matrix()
        for label, value in zip(matrix_mean.index, ova_mean):
            ova_records.append({"seed": seed, "model": name,
                                "class": label, "one_vs_rest_imv": value})
        matrices.setdefault(name, []).append(matrix_mean)

ova = pd.DataFrame(ova_records)
ova.to_csv(RESULTS / f"{DATASET}_one_vs_rest_by_seed.csv", index=False)
mean_matrices = {name: sum(ms) / len(ms) for name, ms in matrices.items()}
for name, matrix in mean_matrices.items():
    matrix.to_csv(RESULTS / f"{DATASET}_pairwise_imv_{name}.csv")
ova.groupby(["model", "class"])["one_vs_rest_imv"].agg(["mean", "std"]).reset_index()


## 5. One-vs-rest figure

Each bar is the mean over the seeds; the error bar is the spread across seeds, which describes stability and is **not** a confidence interval.

One-vs-rest and pairwise IMV are **not on the same scale** and must not be read against each other. The one-vs-rest null model already knows the class base rate, so it starts above chance; pairwise renormalisation leaves its null at the 0.5 chance floor. Identical predictive power therefore reads lower under one-vs-rest.

In [ ]:
ova_summary = (ova.groupby(["model", "class"])["one_vs_rest_imv"]
               .agg(["mean", "std"]).reset_index())
models = list(ova_summary["model"].unique())
fig, axes = plt.subplots(1, len(models), figsize=(5.4 * len(models), 4.6), sharey=True)
axes = np.atleast_1d(axes)
for ax, name in zip(axes, models):
    part = ova_summary[ova_summary.model == name].sort_values("class")
    ax.bar([CLASS_NAMES[int(c)] for c in part["class"]], part["mean"],
           yerr=part["std"].fillna(0), capsize=3,
           color=sns.color_palette("crest", len(part)))
    ax.set_title(f"{TITLE}\n({name})")
    ax.tick_params(axis="x", rotation=45)
    ax.axhline(0, color="0.4", linewidth=0.8)
axes[0].set_ylabel(f"One-vs-rest IMV, mean over {len(SEEDS)} seeds")
fig.suptitle("One-vs-rest IMV (each class against the pooled remainder)", y=1.02)
fig.tight_layout()
paths = {file_format: relative_path(path, start=ARTIFACTS)
         for file_format, path in save_figure(
             fig, FIGURES / f"{DATASET}_one_vs_rest_imv"
         ).items()}
plt.show()
paths

## 6. Pairwise figure

The pairwise matrix is **symmetric by construction** — `ll` is invariant under `(y,p) -> (1-y,1-p)` and pairwise renormalisation gives `p_j = 1 - p_i`. Unlike the ablation matrix it carries no directional information.

In [ ]:
fig, axes = plt.subplots(1, len(mean_matrices), figsize=(5.4 * len(mean_matrices), 4.6))
axes = np.atleast_1d(axes)
for ax, (name, matrix) in zip(axes, mean_matrices.items()):
    sns.heatmap(matrix, annot=True, fmt=".3f", cmap="crest", cbar=False, ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(f"{TITLE}\n({name})")
fig.suptitle("Pairwise IMV (symmetric by construction), mean over "
             f"{len(SEEDS)} seeds", y=1.02)
fig.tight_layout()
paths = {file_format: relative_path(path, start=ARTIFACTS)
         for file_format, path in save_figure(
             fig, FIGURES / f"{DATASET}_pairwise_imv"
         ).items()}
plt.show()
paths
